# Data Preparation
# 数据准备

To fully utilize the power of HftBacktest, it requires to input Tick-by-Tick full order book and trade feed data. Unfortunately, free Tick-by-Tick full order book and trade feed data for HFT is not available unlike daily bar data provided by platforms like Yahoo Finance. However, in the case of cryptocurrency, you can collect the full raw feed yourself.

要充分发挥 HftBacktest 的功能，需要输入逐笔的完整订单簿和交易数据。遗憾的是，与雅虎财经等平台提供的每日柱状图数据不同，用于高频交易的免费逐笔完整订单簿和交易数据并不可用。不过，对于加密货币而言，您可以自行收集完整的原始数据。

## Getting started from Binance Futures' raw feed data
## 从币安期货的原始数据开始操作

You can collect Binance Futures feed yourself using [Data Collector](https://github.com/nkaz001/hftbacktest/tree/master/collector).

您可以自行使用[数据收集器](https://github.com/nkaz001/hftbacktest/tree/master/collector)来获取币安期货的数据。

In [1]:
import gzip

with gzip.open('usdm/ethusdc_20260301.gz', 'r') as f:
    for i in range(5):
        line = f.readline()
        print(line)

b'1772323200009178967 {"stream":"ethusdc@bookTicker","data":{"e":"bookTicker","u":10009695055802,"s":"ETHUSDC","b":"1963.46","B":"2.996","a":"1963.47","A":"2.317","T":1772323200005,"E":1772323200005}}\n'
b'1772323200058390118 {"stream":"ethusdc@depth@0ms","data":{"e":"depthUpdate","E":1772323200056,"T":1772323200036,"s":"ETHUSDC","U":10009695055441,"u":10009695062337,"pu":10009695053845,"b":[["1956.84","0.000"],["1956.98","0.000"],["1959.23","0.023"],["1959.38","6.587"],["1960.07","9.351"],["1961.00","0.258"],["1961.05","0.050"],["1961.88","6.049"],["1962.20","12.649"],["1962.23","12.739"],["1962.31","2.590"],["1962.39","0.022"],["1962.42","2.574"],["1962.96","71.707"],["1963.02","0.000"],["1963.41","0.359"]],"a":[["1963.47","2.317"],["1963.62","0.000"],["1963.98","0.150"],["1964.00","0.425"],["1964.57","5.766"],["1964.63","0.000"],["1964.64","0.431"],["1964.65","0.374"],["1965.98","6.353"],["1966.03","0.364"],["1966.44","0.000"],["1966.51","0.026"]]}}\n'
b'1772323200110978074 {"stream

The first token of the line is timestamp received by local.

这一行的第一个标识是本地接收到的时间戳。

<div class="alert alert-info">
    
**Note:** The timestamp is in nanoseconds.

**Note:** 该时间戳是以纳秒为单位的。
    
</div>

The data needs to be converted to normalized data that can be fed into HftBacktest.  
`convert` method also attempts to correct timestamps by reordering the rows.

这些数据需要转换为可输入到 HftBacktest 中的标准化数据格式。
“转换”方法还会尝试通过重新排列行来修正时间戳。

In [1]:
import numpy as np

from hftbacktest.data.utils import binancefutures

data = binancefutures.convert(
    'usdm/ethusdc_20260301.gz',
    combined_stream=True
)

Correcting the latency
Correcting the event order


Normalized data as follows. You can find more details on [Data](https://hftbacktest.readthedocs.io/en/latest/data.html).

标准化数据如下所示。您可以在[数据](https://hftbacktest.readthedocs.io/en/latest/data.html)页面找到更多详细信息。

In [2]:
import polars as pl

pl.DataFrame(data)

ev,exch_ts,local_ts,px,qty,order_id,ival,fval
u64,i64,i64,f64,f64,u64,i64,f64
3758096385,1772323200036000000,1772323200058390118,1956.84,0.0,0,0,0.0
3758096385,1772323200036000000,1772323200058390118,1956.98,0.0,0,0,0.0
3758096385,1772323200036000000,1772323200058390118,1959.23,0.023,0,0,0.0
3758096385,1772323200036000000,1772323200058390118,1959.38,6.587,0,0,0.0
3758096385,1772323200036000000,1772323200058390118,1960.07,9.351,0,0,0.0
…,…,…,…,…,…,…,…
3489660929,1772409599935000000,1772409599950057813,1940.17,0.0,0,0,0.0
3489660929,1772409599935000000,1772409599950057813,1941.41,1.361,0,0,0.0
3489660929,1772409599935000000,1772409599950057813,1941.9,4.928,0,0,0.0


You can save the data directly to a file by providing `output_filename`.

您可以通过提供“输出文件名”来直接将数据保存到文件中。

In [3]:
_ = binancefutures.convert(
    'usdm/ethusdc_20260301.gz',
    output_filename='usdm/ethusdc_20260301.npz',
    combined_stream=True
)

Correcting the latency
Correcting the event order
Saving to usdm/ethusdc_20260301.npz


## Creating a market depth snapshot
## 创建市场深度概览图

As Binance Futures exchange runs 24/7, you need the initial snapshot to get the complete(almost) market depth.  
[Data Collector](https://github.com/nkaz001/hftbacktest/tree/master/collector) fetches the snapshot only when it makes the connection, so you need build the initial snapshot from the start of the collected feed data.

由于币安期货交易所全天候运行，您需要获取初始快照才能获得完整的（近乎完整的）市场深度。
[数据收集器](https://github.com/nkaz001/hftbacktest/tree/master/collector) 只在建立连接时才获取快照，因此您需要从收集到的原始数据开始构建初始快照。

In [4]:
from hftbacktest.data.utils.snapshot import create_last_snapshot

# Builds 20240808 End of Day snapshot. It will be used for the initial snapshot for 20240809.
data = create_last_snapshot(
    ['usdm/ethusdc_20260301.npz'],
    tick_size=0.1,
    lot_size=0.001
)

Bid levels are shown before ask levels in the snapshot, and levels are sorted from the best price to the farthest price.

在该快照中，卖价水平会排在买价水平之前显示。并且价格水平是按照从最优价格到最远价格的顺序进行排列的。

In [5]:
pl.DataFrame(data)

ev,exch_ts,local_ts,px,qty,order_id,ival,fval
u64,i64,i64,f64,f64,u64,i64,f64
3758096388,0,0,1938.9,0.121,0,0,0.0
3758096388,0,0,1938.8,0.011,0,0,0.0
3758096388,0,0,1938.7,0.011,0,0,0.0
3758096388,0,0,1938.6,6.211,0,0,0.0
3758096388,0,0,1938.5,0.257,0,0,0.0
…,…,…,…,…,…,…,…
3489660932,0,0,4981.7,0.024,0,0,0.0
3489660932,0,0,5000.0,12.95,0,0,0.0
3489660932,0,0,5555.0,2.0,0,0,0.0


In [6]:
from hftbacktest.data.utils.snapshot import create_last_snapshot

# Builds 20240808 End of Day snapshot. It will be used for the initial snapshot for 20240809.
_ = create_last_snapshot(
    ['usdm/ethusdc_20260301.npz'],
    tick_size=0.1,
    lot_size=0.001,
    output_snapshot_filename='usdm/ethusdc_20260301_eod.npz'
)

In [7]:
# Converts 20240809 data.
_ = binancefutures.convert(
    'usdm/ethusdc_20260302.gz',
    output_filename='usdm/ethusdc_20260302.npz',
    combined_stream=True
)

# Builds 20240809's last snapshot.
# Due to the file size limitation of GitHub, ethusdc_20260302.npz does not contain data for the entire day.
_ = create_last_snapshot(
    ['usdm/ethusdc_20260302.npz'],
    tick_size=0.1,
    lot_size=0.001,
    output_snapshot_filename='usdm/ethusdc_20260302_last.npz',
    initial_snapshot='usdm/ethusdc_20260301_eod.npz',
)

Correcting the latency
Correcting the event order
Saving to usdm/ethusdc_20260302.npz


In [8]:
# Builds 20240809's last snapshot without the initial snapshot.
_ = create_last_snapshot(
    ['usdm/ethusdc_20260302.npz'],
    tick_size=0.1,
    lot_size=0.001,
    output_snapshot_filename='usdm/ethusdc_20260302_last_wo_ss.npz'
)

# Builds the 20240809's last snapshot from 20240808 without the initial snapshot.
_ = create_last_snapshot(
    [
        'usdm/ethusdc_20260301.npz',
        'usdm/ethusdc_20260302.npz'
    ],
    tick_size=0.1,
    lot_size=0.001,
    output_snapshot_filename='usdm/ethusdc_20260302_last.npz'
)

## Getting started from Tardis.dev data
## 从 Tardis.dev 数据开始操作

Few vendors offer tick-by-tick full market depth data along with snapshot and trade data, and Tardis.dev is among them.

很少有供应商能同时提供逐笔成交的完整市场深度数据、快照数据以及交易数据，而 Tardis.dev 就是其中之一。

<div class="alert alert-info">
    
**Note:** Some data may have an issue with the exchange timestamp. Ideally, the exchange timestamp should reflect the moment the event occurs at the matching engine. However, some data uses the server's data sent timestamp instead of the matching engine timestamp.

**Note:** 有些数据可能存在交换时间戳的问题。理想情况下，交换时间戳应反映事件在匹配引擎中发生的具体时刻。然而，有些数据使用的却是服务器发送的数据时间戳，而非匹配引擎的时间戳。

</div>

In [10]:
# https://docs.tardis.dev/historical-data-details/binance-futures

# Downloads sample Binance futures BTCUSDT trades
!wget https://datasets.tardis.dev/v1/binance-futures/trades/2020/02/01/BTCUSDT.csv.gz -O BTCUSDT_trades.csv.gz
    
# Downloads sample Binance futures BTCUSDT book
!wget https://datasets.tardis.dev/v1/binance-futures/incremental_book_L2/2020/02/01/BTCUSDT.csv.gz -O BTCUSDT_book.csv.gz

--2024-08-09 09:42:51--  https://datasets.tardis.dev/v1/binance-futures/trades/2020/02/01/BTCUSDT.csv.gz
Resolving datasets.tardis.dev (datasets.tardis.dev)... 104.18.6.96, 104.18.7.96, 2606:4700::6812:760, ...
Connecting to datasets.tardis.dev (datasets.tardis.dev)|104.18.6.96|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3090479 (2.9M) [text/csv]
Saving to: ‘BTCUSDT_trades.csv.gz’

BTCUSDT_trades.csv. 100%[===================>]   2.95M  5.66MB/s    in 0.5s    

2024-08-09 09:42:52 (5.66 MB/s) - ‘BTCUSDT_trades.csv.gz’ saved [3090479/3090479]

--2024-08-09 09:42:52--  https://datasets.tardis.dev/v1/binance-futures/incremental_book_L2/2020/02/01/BTCUSDT.csv.gz
Resolving datasets.tardis.dev (datasets.tardis.dev)... 104.18.7.96, 104.18.6.96, 2606:4700::6812:760, ...
Connecting to datasets.tardis.dev (datasets.tardis.dev)|104.18.7.96|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 250016849 (238M) [text/csv]
Saving to: ‘BTCUSDT_book.cs

It is recommended to input trade files before depth files. This is because if a depth event occurs due to a trade event, having the trade event before the depth event could provide a more realistic fill during backtesting. However, the sorting process will prioritize events from the first input file when both events have the same timestamp.

建议先输入交易文件，然后再输入深度文件。这是因为如果由于交易事件而产生了深度事件，那么将交易事件置于深度事件之前，能够在回测过程中提供更真实的成交情况。然而，当两个事件的时间戳相同时，排序过程会优先处理第一个输入文件中的事件。

In [11]:
from hftbacktest.data.utils import tardis

data = tardis.convert(
    ['BTCUSDT_trades.csv.gz', 'BTCUSDT_book.csv.gz']
)

Reading BTCUSDT_trades.csv.gz
Reading BTCUSDT_book.csv.gz
Correcting the latency
Correcting the event order


In [12]:
pl.DataFrame(data)

ev,exch_ts,local_ts,px,qty,order_id,ival,fval
u64,i64,i64,f64,f64,u64,i64,f64
3758096386,1580515202342000000,1580515202497052000,9364.51,1.197,0,0,0.0
3758096386,1580515202342000000,1580515202497346000,9365.67,0.02,0,0,0.0
3758096386,1580515202342000000,1580515202497352000,9365.86,0.01,0,0,0.0
3758096386,1580515202342000000,1580515202497357000,9366.36,0.002,0,0,0.0
3758096386,1580515202342000000,1580515202497363000,9366.36,0.003,0,0,0.0
…,…,…,…,…,…,…,…
3489660929,1580601599812000000,1580601599944404000,9397.79,0.0,0,0,0.0
3758096385,1580601599826000000,1580601599952176000,9354.8,4.07,0,0,0.0
3758096385,1580601599836000000,1580601599962961000,9351.47,3.914,0,0,0.0


You can save the data directly to a file by providing `output_filename`. If there are too many rows, you need to increase `buffer_size`.  

您可以通过提供“输出文件名”来直接将数据保存到文件中。如果数据行数过多，您需要增加“缓冲区大小”。

In [13]:
_ = tardis.convert(
    ['BTCUSDT_trades.csv.gz', 'BTCUSDT_book.csv.gz'],
    output_filename='btcusdt_20200201.npz',
    buffer_size=200_000_000
)

Reading BTCUSDT_trades.csv.gz
Reading BTCUSDT_book.csv.gz
Correcting the latency
Correcting the event order
Saving to btcusdt_20200201.npz


Tardis.dev artificially inserts the SOD snapshot to the start of the daily file. If you continuously backtest multiple days, you don't need the snapshot every start of days and it may incur more time to backtest. You can choose to include the Tardis.dev's SOD snapshot in the converted file using the option.

Tardis.dev 会人为地将 SOD 快照插入到每日文件的开头位置。如果您连续进行多天的回测，那么在每天开始时您无需每次都使用该快照，这样可能会导致回测所需时间增加。您可以选择通过该选项将 Tardis.dev 的 SOD 快照包含在转换后的文件中。